In [1]:
import os
from pathlib import Path

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\jarvis\Desktop\NextWord-AI\notebooks


In [2]:
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)
print("Folders:", [p.name for p in PROJECT_ROOT.iterdir()])

Project root: C:\Users\jarvis\Desktop\NextWord-AI
Folders: ['.gitignore', 'app.py', 'data', 'models', 'notebooks', 'README.md', 'requirements.txt', 'src']


In [3]:
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "books"

ebook_path = RAW_DATA_PATH / "pride_prejudice.txt"

print("Dataset exists:", ebook_path.exists())

if ebook_path.exists():
    print("File size:", round(ebook_path.stat().st_size / 1024, 2), "KB")

Dataset exists: False


In [4]:
%pip install requests pandas numpy matplotlib seaborn nltk tensorflow

Note: you may need to restart the kernel to use updated packages.


In [5]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [6]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Current directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Git initialized:", (PROJECT_ROOT / ".git").exists())

Current directory: C:\Users\jarvis\Desktop\NextWord-AI\notebooks
Project root: C:\Users\jarvis\Desktop\NextWord-AI
Git initialized: False


In [7]:
!git -C .. init

Initialized empty Git repository in C:/Users/jarvis/Desktop/NextWord-AI/.git/


In [8]:
!git -C .. status

On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	README.md
	app.py
	data/
	notebooks/
	requirements.txt
	src/

nothing added to commit but untracked files present (use "git add" to track)


In [9]:
!git -C .. branch -M main

In [10]:
!git -C .. add .
!git -C .. status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   .gitignore
	new file:   README.md
	new file:   app.py
	new file:   data/interim/.gitkeep
	new file:   data/processed/.gitkeep
	new file:   data/raw/.gitkeep
	new file:   notebooks/01_data_collection.ipynb
	new file:   notebooks/02_data_cleaning.ipynb
	new file:   notebooks/03_eda.ipynb
	new file:   notebooks/04_preprocessing.ipynb
	new file:   notebooks/05_model_training.ipynb
	new file:   requirements.txt
	new file:   src/cleaning.py
	new file:   src/model.py
	new file:   src/prediction.py
	new file:   src/preprocessing.py



In [11]:
!git -C .. commit -m "chore: initialize NextWord-AI project"

[main (root-commit) ecaaca5] chore: initialize NextWord-AI project
 16 files changed, 307 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 README.md
 create mode 100644 app.py
 create mode 100644 data/interim/.gitkeep
 create mode 100644 data/processed/.gitkeep
 create mode 100644 data/raw/.gitkeep
 create mode 100644 notebooks/01_data_collection.ipynb
 create mode 100644 notebooks/02_data_cleaning.ipynb
 create mode 100644 notebooks/03_eda.ipynb
 create mode 100644 notebooks/04_preprocessing.ipynb
 create mode 100644 notebooks/05_model_training.ipynb
 create mode 100644 requirements.txt
 create mode 100644 src/cleaning.py
 create mode 100644 src/model.py
 create mode 100644 src/prediction.py
 create mode 100644 src/preprocessing.py


In [25]:
!git -C .. remote set-url origin https://github.com/triyanjha04-dev/NextWord-AI.git

In [26]:
!git -C .. remote -v

origin	https://github.com/triyanjha04-dev/NextWord-AI.git (fetch)
origin	https://github.com/triyanjha04-dev/NextWord-AI.git (push)


In [27]:
!git -C .. push -u origin main

To https://github.com/triyanjha04-dev/NextWord-AI.git
 ! [rejected]        main -> main (fetch first)
error: failed to push some refs to 'https://github.com/triyanjha04-dev/NextWord-AI.git'
hint: Updates were rejected because the remote contains work that you do not
hint: have locally. This is usually caused by another repository pushing to
hint: the same ref. If you want to integrate the remote changes, use
hint: 'git pull' before pushing again.
hint: See the 'Note about fast-forwards' in 'git push --help' for details.


In [28]:
!git -C .. fetch origin

From https://github.com/triyanjha04-dev/NextWord-AI
 * [new branch]      main       -> origin/main


In [29]:
!git -C .. pull origin main --allow-unrelated-histories

Merge made by the 'ort' strategy.
 .../extracting data-checkpoint.ipynb               |     6 +
 extracting data.ipynb                              |  1007 ++
 pride_prejudice.txt                                | 14915 +++++++++++++++++++
 3 files changed, 15928 insertions(+)
 create mode 100644 .ipynb_checkpoints/extracting data-checkpoint.ipynb
 create mode 100644 extracting data.ipynb
 create mode 100644 pride_prejudice.txt


From https://github.com/triyanjha04-dev/NextWord-AI
 * branch            main       -> FETCH_HEAD


In [30]:
!git -C .. status

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_data_collection.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/.~01_data_collection.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [31]:
!git -C .. push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/triyanjha04-dev/NextWord-AI.git
   d0a01d9..2ca9021  main -> main


In [41]:
import requests
import json
import time
import re
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent

BOOKS_PATH = PROJECT_ROOT / "data" / "raw" / "books"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "book_metadata.json"

BOOKS_PATH.mkdir(parents=True, exist_ok=True)

print("Books directory:", BOOKS_PATH)

Books directory: C:\Users\jarvis\Desktop\NextWord-AI\data\raw\books


In [44]:
def download_new_books(max_books=5):

    # Load previously downloaded book records
    if METADATA_PATH.exists():
        metadata = json.loads(
            METADATA_PATH.read_text(encoding="utf-8")
        )
    else:
        metadata = {}

    downloaded_count = 0
    skipped_count = 0

    next_url = API_URL

    params = {
        "languages": "en",
        "copyright": "false",
        "mime_type": "text/plain",
        "sort": "descending"
    }

    session = requests.Session()
    session.headers.update({
        "User-Agent": "NextWord-AI educational research project"
    })

    while next_url and downloaded_count < max_books:

        response = session.get(
            next_url,
            params=params if next_url == API_URL else None,
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        # Follow pagination
        next_url = data["next"]

        for book in data["results"]:

            book_id = str(book["id"])
            file_path = BOOKS_PATH / f"{book_id}.txt"

            # Skip previously downloaded books
            if book_id in metadata or file_path.exists():
                skipped_count += 1
                continue

            # Find a plain-text download URL
            formats = book.get("formats", {})

            text_url = next(
                (
                    url for file_type, url in formats.items()
                    if file_type.lower().startswith("text/plain")
                    and not url.lower().endswith(".zip")
                ),
                None
            )

            if not text_url:
                continue

            try:
                print(f"Downloading: {book['title']}")

                book_response = session.get(
                    text_url,
                    timeout=60
                )

                book_response.raise_for_status()

                book_text = book_response.content.decode(
                    "utf-8",
                    errors="replace"
                )

                # Reject empty or very small files
                if len(book_text) < 1000:
                    print("Skipped: insufficient text")
                    continue

                # Save raw text
                file_path.write_text(
                    book_text,
                    encoding="utf-8"
                )

                # Save book information
                metadata[book_id] = {
                    "id": book["id"],
                    "title": book["title"],
                    "authors": [
                        author["name"]
                        for author in book.get("authors", [])
                    ],
                    "languages": book.get("languages", []),
                    "copyright": book.get("copyright"),
                    "download_count": book.get("download_count"),
                    "source_url": text_url,
                    "downloaded_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "file": file_path.name
                }

                # Save metadata immediately after each successful download
                METADATA_PATH.write_text(
                    json.dumps(metadata, indent=4, ensure_ascii=False),
                    encoding="utf-8"
                )

                downloaded_count += 1

                print(f"Saved: {file_path.name}")
                print(f"Characters: {len(book_text):,}")

                if downloaded_count >= max_books:
                    break

                # Be considerate to the source server
                time.sleep(1)

            except requests.RequestException as error:
                print(f"Download failed for {book_id}: {error}")

        params = None

    print("\nDownload session completed!")
    print("New books downloaded:", downloaded_count)
    print("Previously downloaded books skipped:", skipped_count)
    print("Total books in catalogue:", len(metadata))

In [45]:
download_new_books(max_books=5)

Downloading: Geoff the djinn
Saved: 79971.txt
Characters: 33,801
Downloading: Inside the shell
Saved: 79969.txt
Characters: 38,250
Downloading: Scenes in the lives of Saint Philip, Saint Bartholomew, and Saint Thomas : $b The good child's library, seventh book. In easy verse
Saved: 79968.txt
Characters: 26,101
Downloading: Home and garden : $b Notes and thoughts, practical and critical, of a worker in both
Saved: 79967.txt
Characters: 432,901
Downloading: Blue law ballads : $b A purge for Puritans
Saved: 79965.txt
Characters: 66,884

Download session completed!
New books downloaded: 5
Previously downloaded books skipped: 0
Total books in catalogue: 5


In [46]:
download_new_books(max_books=5)

Downloading: Mothers still
Saved: 79964.txt
Characters: 12,781
Downloading: A Pennsylvania bison hunt : $b being the results of an investigation into the causes and period of the destruction of these noble beasts in the Keystone state, obtained from descendants of the original hunters. Including a sketch of the career of Daniel Ott, a Pennsylvanian who has killed many buffaloes in the West.
Saved: 79963.txt
Characters: 68,482
Downloading: The hallway : $b A significant episode in one scene
Saved: 79962.txt
Characters: 6,843
Downloading: ...Divided we fall
Saved: 79961.txt
Characters: 187,662
Downloading: The mysterious Miss Morrisot
Saved: 79960.txt
Characters: 444,570

Download session completed!
New books downloaded: 5
Previously downloaded books skipped: 5
Total books in catalogue: 10


In [47]:
!git -C .. add notebooks/01_data_collection.ipynb
!git -C .. commit -m "feat: build automated ebook downloader"
!git -C .. push

[main c3acdf9] feat: build automated ebook downloader
 1 file changed, 179 insertions(+), 139 deletions(-)


To https://github.com/triyanjha04-dev/NextWord-AI.git
   e119f87..c3acdf9  main -> main


In [48]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"

SRC_PATH.mkdir(parents=True, exist_ok=True)

print("Source directory:", SRC_PATH)

Source directory: C:\Users\jarvis\Desktop\NextWord-AI\src


In [49]:
%%writefile ../src/data_collection.py

import requests
import json
import time

from pathlib import Path
from datetime import datetime, timezone


PROJECT_ROOT = Path(__file__).resolve().parent.parent

BOOKS_PATH = PROJECT_ROOT / "data" / "raw" / "books"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "book_metadata.json"

API_URL = "https://gutendex.com/books/"


def load_metadata():

    if METADATA_PATH.exists():
        return json.loads(
            METADATA_PATH.read_text(encoding="utf-8")
        )

    return {}


def save_metadata(metadata):

    METADATA_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    METADATA_PATH.write_text(
        json.dumps(metadata, indent=4, ensure_ascii=False),
        encoding="utf-8"
    )


def download_new_books(max_books=5):

    BOOKS_PATH.mkdir(parents=True, exist_ok=True)

    metadata = load_metadata()

    downloaded_count = 0
    skipped_count = 0

    params = {
        "languages": "en",
        "copyright": "false",
        "mime_type": "text/plain",
        "sort": "descending"
    }

    next_url = API_URL

    session = requests.Session()
    session.headers.update({
        "User-Agent": "NextWord-AI educational research project"
    })

    while next_url and downloaded_count < max_books:

        response = session.get(
            next_url,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()
        next_url = data["next"]
        params = None

        for book in data["results"]:

            book_id = str(book["id"])
            file_path = BOOKS_PATH / f"{book_id}.txt"

            if book_id in metadata or file_path.exists():
                skipped_count += 1
                continue

            formats = book.get("formats", {})

            text_url = next(
                (
                    url for file_type, url in formats.items()
                    if file_type.lower().startswith("text/plain")
                    and not url.lower().endswith(".zip")
                ),
                None
            )

            if not text_url:
                continue

            try:

                print("Downloading:", book["title"])

                book_response = session.get(
                    text_url,
                    timeout=60
                )

                book_response.raise_for_status()

                book_text = book_response.content.decode(
                    "utf-8",
                    errors="replace"
                )

                if len(book_text) < 1000:
                    print("Skipped: insufficient text")
                    continue

                file_path.write_text(
                    book_text,
                    encoding="utf-8"
                )

                metadata[book_id] = {
                    "id": book["id"],
                    "title": book["title"],
                    "authors": [
                        author["name"]
                        for author in book.get("authors", [])
                    ],
                    "languages": book.get("languages", []),
                    "copyright": book.get("copyright"),
                    "download_count": book.get("download_count"),
                    "source_url": text_url,
                    "downloaded_at": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "file": file_path.name
                }

                save_metadata(metadata)

                downloaded_count += 1

                print("Saved:", file_path.name)

                if downloaded_count >= max_books:
                    break

                time.sleep(1)

            except requests.RequestException as error:

                print(f"Download failed for {book_id}: {error}")

    print("\nDownload completed!")
    print("New downloads:", downloaded_count)
    print("Previously downloaded books skipped:", skipped_count)
    print("Total catalogue records:", len(metadata))

Writing ../src/data_collection.py


In [50]:
import sys
import importlib

sys.path.append(str(PROJECT_ROOT))

import src.data_collection as collector

importlib.reload(collector)

<module 'src.data_collection' from 'C:\\Users\\jarvis\\Desktop\\NextWord-AI\\src\\data_collection.py'>

In [51]:
collector.download_new_books(max_books=5)

Downloading: The girl in scarlet : $b or, The loves of Silvère and Miette.
Saved: 79959.txt
Downloading: The pink puppy dog
Saved: 79958.txt
Downloading: The complete poetical works of Geoffrey Chaucer, now first put into modern English
Saved: 79952.txt
Downloading: First lessons in Geometry
Saved: 79951.txt
Downloading: Scenes in the lives of Saint Stephen, Timothy, Saint Mark, and Saint Luke, in easy verse
Saved: 79950.txt

Download completed!
New downloads: 5
Previously downloaded books skipped: 10
Total catalogue records: 15
